# Пример разбиения на группы (AASplitter)

В этом ноутбуке показано, как работает `AASplitter` — базовый сплиттер для разбиения данных на control/test группы.

In [1]:
from hypex import AATest
from hypex.dataset import (
    Dataset,
    SmallDataset,
    InfoRole,
    TargetRole,
    TreatmentRole,
    DatasetAdapter,
    DefaultRole,
    StratificationRole
)
from hypex.dataset.roles import TargetRole, FeatureRole, InfoRole, StatisticRole
from hypex.comparators import StatsTTest
from hypex.utils import create_test_data
from hypex.splitters import AASplitter
from hypex.comparators import GroupSizes
import pyspark.sql as spark
import pandas as pd

/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/__init__.py:50: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(
26/03/24 16:37:32 WARN Utils: Your hostname, MacBook-Pro-Danil.local resolves to a loopback address: 127.0.0.1; using 10.240.107.179 instead (on interface en0)
26/03/24 16:37:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/24 16:37:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
DatasetAdapter.to_dataset(
                {'mean┆pre_spends': 487.30713810684995, 'std┆pre_spends': 18.70852225356039, 'count┆pre_spends': 4511},
                StatisticRole(),
            )

from_dict data = {'mean┆pre_spends': 487.30713810684995, 'std┆pre_spends': 18.70852225356039, 'count┆pre_spends': 4511}
BackendsEnum.pandas
pandas_backend dict = {'data': {'mean┆pre_spends': 487.30713810684995, 'std┆pre_spends': 18.70852225356039, 'count┆pre_spends': 4511}}
out =    mean┆pre_spends  std┆pre_spends  count┆pre_spends
0       487.307138       18.708522              4511
data = {'data': {'mean┆pre_spends': 487.30713810684995, 'std┆pre_spends': 18.70852225356039, 'count┆pre_spends': 4511}}
pandas_backend dict = {'data': {'mean┆pre_spends': 487.30713810684995, 'std┆pre_spends': 18.70852225356039, 'count┆pre_spends': 4511}}


,mean┆pre_spends,std┆pre_spends,count┆pre_spends
0,487.307138,18.708522,4511.0


## 1. Создание тестового датасета

Создадим синтетические данные с пользователями и их метриками.

In [3]:
session = (
            spark
            .SparkSession.builder
            .master("local[*]")
            .config("spark.driver.memory", "4g")
            .config("spark.executo|r.memory", "4g")
            .config("spark.memory.fraction", "0.8") 
            .config("spark.memory.storageFraction", "0.3") 
            .getOrCreate()
          )
data = (
            session.read.format('csv')
            .option("header", "true")
            .option("inferSchema", "true")
            .option("ignoreLeadingWhiteSpace", "true") 
            .option("ignoreTrailingWhiteSpace", "true")
            .load("./data_no_index.csv")
        )
# data = pd.read_csv("../data_no_index.csv")
ds = Dataset(
    roles={
        "user_id": InfoRole(float),
        "treat": TreatmentRole(),
        "pre_spends": TargetRole(),
        "post_spends": DefaultRole(),
        "gender": StratificationRole(str),
        "industry" : DefaultRole()
    }, 
    data=data, 
    session=session
    )

ds

26/03/24 16:37:34 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warning

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry
0,0.0,0.0,0.0,519.5,424.666667,39.0,M,Logistics
1,1.0,0.0,0.0,500.5,423.111111,19.0,F,Logistics
2,2.0,0.0,0.0,499.0,423.444444,29.0,M,E-commerce
3,3.0,2.0,1.0,494.0,530.555556,38.0,M,E-commerce
4,4.0,9.0,1.0,455.0,451.888889,53.0,F,Logistics
...,...,...,...,...,...,...,...,...
9995,9995.0,0.0,0.0,490.0,429.555556,26.0,F,Logistics
9996,9996.0,0.0,0.0,464.5,414.666667,49.0,M,E-commerce
9997,9997.0,0.0,0.0,498.0,411.666667,30.0,F,E-commerce
9998,9998.0,2.0,1.0,528.0,516.555556,36.0,F,Logistics


## 2. Запуск AASplitter

`AASplitter` создаёт колонку с метками групп (control, test_0, test_1, ...).

Параметры:
- `control_size` — доля контрольной группы (по умолчанию 0.5)
- `random_state` — seed для воспроизводимости
- `sample_size` — доля данных для эксперимента (по умолчанию 1.0)

In [4]:
from hypex.dataset import ExperimentData, AdditionalTreatmentRole

# Оборачиваем в ExperimentData для работы пайплайна
exp_data = ExperimentData(ds)

# Создаём и запускаем сплиттер
splitter = AASplitter(
    control_size=0.5,
    random_state=21,
    constant_key=True,
    save_groups=True  # Сохраняем группы в data.groups для удобного доступа
)
result = splitter.execute(exp_data)

print("Разбиение выполнено!")
print(f"ID сплиттера: {splitter.id}")

BackendsEnum.spark


/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


Разбиение выполнено!
ID сплиттера: AASplitter┴rs 21┴


## 3. Просмотр результатов разбиения

Посмотрим на созданную колонку с метками групп и размеры групп.

In [5]:
# Колонка с метками групп сохраняется в additional_fields
split_column = result.additional_fields[splitter.id]
print("Колонка с метками групп:")
print(split_column.value_counts())

Колонка с метками групп:


/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/base.py:1437: FutureWarning: The resulting Series will have a fixed name of 'count' from 4.0.0.
  warnings.warn(
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


  AASplitter┴rs 21┴  count
0           control   5000
1            test_0   5000

2 rows × 2 columns


In [6]:
result._data

/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.1

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry
0,0.0,0.0,0.0,519.5,424.666667,39.0,M,Logistics
1,1.0,0.0,0.0,500.5,423.111111,19.0,F,Logistics
2,2.0,0.0,0.0,499.0,423.444444,29.0,M,E-commerce
3,3.0,2.0,1.0,494.0,530.555556,38.0,M,E-commerce
4,4.0,9.0,1.0,455.0,451.888889,53.0,F,Logistics
...,...,...,...,...,...,...,...,...
9995,9995.0,0.0,0.0,490.0,429.555556,26.0,F,Logistics
9996,9996.0,0.0,0.0,464.5,414.666667,49.0,M,E-commerce
9997,9997.0,0.0,0.0,498.0,411.666667,30.0,F,E-commerce
9998,9998.0,2.0,1.0,528.0,516.555556,36.0,F,Logistics


In [7]:
# # Используем GroupSizes для подсчёта размеров групп
# group_sizes = GroupSizes(grouping_role=AdditionalTreatmentRole())
# sizes_result = group_sizes.execute(result)

# print("\nРазмеры групп:")
# print(sizes_result.analysis_tables[group_sizes.id])

In [8]:
result.ds.to_small_dataset()

/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


BackendsEnum.pandas
out =       user_id  signup_month  treat  pre_spends  post_spends   age gender  \
0         0.0           0.0    0.0       519.5   424.666667  39.0      M   
1         1.0           0.0    0.0       500.5   423.111111  19.0      F   
2         2.0           0.0    0.0       499.0   423.444444  29.0      M   
3         3.0           2.0    1.0       494.0   530.555556  38.0      M   
4         4.0           9.0    1.0       455.0   451.888889  53.0      F   
...       ...           ...    ...         ...          ...   ...    ...   
9995   9995.0           0.0    0.0       490.0   429.555556  26.0      F   
9996   9996.0           0.0    0.0       464.5   414.666667  49.0      M   
9997   9997.0           0.0    0.0       498.0   411.666667  30.0      F   
9998   9998.0           2.0    1.0       528.0   516.555556  36.0      F   
9999   9999.0           7.0    1.0       507.5   471.333333  28.0      F   

        industry  
0      Logistics  
1      Logistics  
2   

/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry
0,0.0,0.0,0.0,519.5,424.666667,39.0,M,Logistics
1,1.0,0.0,0.0,500.5,423.111111,19.0,F,Logistics
2,2.0,0.0,0.0,499.0,423.444444,29.0,M,E-commerce
3,3.0,2.0,1.0,494.0,530.555556,38.0,M,E-commerce
4,4.0,9.0,1.0,455.0,451.888889,53.0,F,Logistics
...,...,...,...,...,...,...,...,...
9995,9995.0,0.0,0.0,490.0,429.555556,26.0,F,Logistics
9996,9996.0,0.0,0.0,464.5,414.666667,49.0,M,E-commerce
9997,9997.0,0.0,0.0,498.0,411.666667,30.0,F,E-commerce
9998,9998.0,2.0,1.0,528.0,516.555556,36.0,F,Logistics


# Тестируем стат.тесты

In [9]:
test = StatsTTest(
    grouping_role=AdditionalTreatmentRole(),
    target_roles=TargetRole(),
    reliability=0.05
)
result_test = test.execute(result)

BackendsEnum.spark


/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


BackendsEnum.spark
BackendsEnum.spark


/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarn

                   pre_spends┆mean  pre_spends┆std  pre_spends┆count
AASplitter┴rs 21┴                                                   
control                 487.307138       18.708522              4511
test_0                  487.563252       19.322387              4490

2 rows × 3 columns
['control', 'test_0']
[{'mean┆pre_spends': 487.30713810684995, 'std┆pre_spends': 18.70852225356039, 'count┆pre_spends': 4511}, {'mean┆pre_spends': 487.56325167037863, 'std┆pre_spends': 19.322387347936747, 'count┆pre_spends': 4490}]
from_dict data = {'mean┆pre_spends': 487.30713810684995, 'std┆pre_spends': 18.70852225356039, 'count┆pre_spends': 4511}
BackendsEnum.pandas
pandas_backend dict = {'data': {'mean┆pre_spends': 487.30713810684995, 'std┆pre_spends': 18.70852225356039, 'count┆pre_spends': 4511}}
out =    mean┆pre_spends  std┆pre_spends  count┆pre_spends
0       487.307138       18.708522              4511
data = {'data': {'mean┆pre_spends': 487.30713810684995, 'std┆pre_spends': 18.70852225

In [26]:
result_test.analysis_tables["StatsTTest┴┴pre_spends"]

,p-value,statistic,pass
test_0┆pre_spends,0.005345,2.786131,1.0


In [23]:
pd.DataFrame([{'mean┆pre_spends': 487.30713810684995, 'std┆pre_spends': 18.70852225356039, 'count┆pre_spends': 4511}])

,mean┆pre_spends,std┆pre_spends,count┆pre_spends
0,487.307138,18.708522,4511


In [20]:
DatasetAdapter.to_dataset(
                {'mean┆pre_spends': [487.30713810684995], 'std┆pre_spends': [18.70852225356039], 'count┆pre_spends': [4511]},
                StatisticRole(),
            )

from_dict data = {'mean┆pre_spends': [487.30713810684995], 'std┆pre_spends': [18.70852225356039], 'count┆pre_spends': [4511]}
BackendsEnum.pandas
pandas_backend dict = {'data': {'mean┆pre_spends': [487.30713810684995], 'std┆pre_spends': [18.70852225356039], 'count┆pre_spends': [4511]}}
out =         mean┆pre_spends       std┆pre_spends count┆pre_spends
0  [487.30713810684995]  [18.70852225356039]           [4511]
data = {'data': {'mean┆pre_spends': [487.30713810684995], 'std┆pre_spends': [18.70852225356039], 'count┆pre_spends': [4511]}}
pandas_backend dict = {'data': {'mean┆pre_spends': [487.30713810684995], 'std┆pre_spends': [18.70852225356039], 'count┆pre_spends': [4511]}}


,mean┆pre_spends,std┆pre_spends,count┆pre_spends
0,[487.30713810684995],[18.70852225356039],[4511]


In [11]:
data = {'data': {'mean┆pre_spends': 487.30713810684995, 'std┆pre_spends': 18.70852225356039, 'count┆pre_spends': 4511}}
pd.DataFrame(data=[data["data"]])

,mean┆pre_spends,std┆pre_spends,count┆pre_spends
0,487.307138,18.708522,4511


In [12]:
DatasetAdapter.to_dataset(
                [{'mean┆pre_spends': 487.30713810684995, 'std┆pre_spends': 18.70852225356039, 'count┆pre_spends': 4511}, {'mean┆pre_spends': 487.56325167037863, 'std┆pre_spends': 19.322387347936747, 'count┆pre_spends': 4490}][0],
                StatisticRole(),
            )

from_dict data = {'mean┆pre_spends': 487.30713810684995, 'std┆pre_spends': 18.70852225356039, 'count┆pre_spends': 4511}
BackendsEnum.pandas
pandas_backend dict = {'data': {'mean┆pre_spends': 487.30713810684995, 'std┆pre_spends': 18.70852225356039, 'count┆pre_spends': 4511}}
out =    mean┆pre_spends  std┆pre_spends  count┆pre_spends
0       487.307138       18.708522              4511
data = {'data': {'mean┆pre_spends': 487.30713810684995, 'std┆pre_spends': 18.70852225356039, 'count┆pre_spends': 4511}}
pandas_backend dict = {'data': {'mean┆pre_spends': 487.30713810684995, 'std┆pre_spends': 18.70852225356039, 'count┆pre_spends': 4511}}


,mean┆pre_spends,std┆pre_spends,count┆pre_spends
0,487.307138,18.708522,4511.0


In [13]:
d = {'backend': 'sparkdataset', 
     'roles': {'role_names': ['pre_spends┆mean', 'pre_spends┆std', 'pre_spends┆count']}, 
     'data': {'pre_spends┆mean': {'control': 487.30713810684995, 'test_0': 487.56325167037863}, 
              'pre_spends┆std': {'control': 18.70852225356039, 'test_0': 19.322387347936747}, 
              'pre_spends┆count': {'control': 4511, 'test_0': 4490}}}
d

{'backend': 'sparkdataset',
 'roles': {'role_names': ['pre_spends┆mean',
   'pre_spends┆std',
   'pre_spends┆count']},
 'data': {'pre_spends┆mean': {'control': 487.30713810684995,
   'test_0': 487.56325167037863},
  'pre_spends┆std': {'control': 18.70852225356039,
   'test_0': 19.322387347936747},
  'pre_spends┆count': {'control': 4511, 'test_0': 4490}}}

In [14]:
return {
            str(raw): {
                col: {
                    stat: agg_ds.get_values(
                        row=raw, column=f"{col}{NAME_BORDER_SYMBOL}{stat}"
                    )
                    for stat in stats
                }
                for col in target_columns
            }
            for raw in raw_group_names
        }

SyntaxError: 'return' outside function (2195841108.py, line 1)

In [ ]:
list(next(iter(d["data"].values())).keys())

In [ ]:
def transform_stats_format(data):
    """
    Преобразует словарь из формата {column┆stat: {group: value}}
    в формат {group: {column: {stat: value}}}
    """
    result = {}
    
    for col_stat, groups in data.items():
        # Разделяем column и stat по разделителю ┆
        column, stat = col_stat.split('┆')
        
        for group, value in groups.items():
            # Инициализируем структуру если нужно
            if group not in result:
                result[group] = {}
            if column not in result[group]:
                result[group][column] = {}
            
            # Добавляем значение
            result[group][column][stat] = value
    
    return result

In [ ]:
transform_stats_format(d["data"])

In [ ]:
d["data"]

In [ ]:
for x in d["data"].values():
    print(x)

In [ ]:
# Просмотр результатов т-теста
print("Результаты т-теста:")
for table_id, table in result_test.analysis_tables.items():
    print(f"\nТаблица {table_id}:")
    print(table)

## 4. Доступ к группам

Если `save_groups=True`, группы сохраняются в `result.groups` для удобного доступа.

In [ ]:
# Прямой доступ к группам
print("Доступные группы:", list(result.groups.keys()))

# Получаем данные по имени группы
control_group = result.groups[splitter.id]['control']
test_group = result.groups[splitter.id]['test_0']

print(f"\nControl группа: {len(control_group)} записей")
print(f"Test группа: {len(test_group)} записей")

In [ ]:
# Сравниваем средние значения метрик между группами
print("\nСредние значения в control группе:")
print(control_group[['pre_spends', 'post_spends']].mean())

print("\nСредние значения в test группе:")
print(test_group[['pre_spends', 'post_spends']].mean())

## 5. Несколько разбиений с разными random_state

Покажем, как можно запустить несколько итераций с разными seed.

In [ ]:
# Запускаем 5 разбиений с разными random_state
for rs in [10, 20, 30, 40, 50]:
    splitter = AASplitter(control_size=0.5, random_state=rs, save_groups=False)
    result = splitter.execute(ExperimentData(data))
    
    split_col = result.additional_fields[splitter.id]
    counts = split_col.value_counts()
    
    print(f"random_state={rs}: control={counts.get('control', 0)}, test_0={counts.get('test_0', 0)}")

## 6. Разбиение на несколько групп

Можно указать `groups_sizes` для разбиения на более чем 2 группы.

In [ ]:
# Разбиение на 3 группы: 50% control, 30% test_0, 20% test_1
splitter = AASplitter(
    groups_sizes=[0.5, 0.3, 0.2],
    random_state=42,
    save_groups=True
)
result = splitter.execute(ExperimentData(data))

split_col = result.additional_fields[splitter.id]
print("Разбиение на 3 группы:")
print(split_col.value_counts())

print("\nДоступные группы:", list(result.groups[splitter.id].keys()))

## Итоги

`AASplitter` выполняет:
1. Создаёт колонку с метками групп в `additional_fields`
2. Опционально сохраняет группы в `data.groups` для удобного доступа
3. Поддерживает кастомные размеры групп через `groups_sizes`
4. Используется как первый шаг в AA/AB тестах